# Part 2: Text Analysis — LSA + OpenAI gpt-4o-mini

**Course:** CSIS 4260 — Douglas College  
**Student:** Desmond Chua  

In this notebook, we apply two text summarization algorithms to the 100 healthcare IT posts scraped in Part 1.

**Algorithm 1 — LSA (Latent Semantic Analysis):**  
An extractive summarization method that runs locally. It uses linear algebra to identify the most important sentences in each post and pulls them out verbatim. No API key required — it's free and fast.

**Algorithm 2 — OpenAI gpt-4o-mini:**  
An abstractive summarization method that uses OpenAI's API. It generates new sentences that capture the meaning of the original text, and can be prompted to focus on healthcare-specific context. Requires an API key.

By comparing both approaches, we can see the trade-offs between extractive (LSA) and abstractive (GPT) summarization on real healthcare forum data.

In [ ]:
import pandas as pd

# Load the scraped data from Part 1
# We use ../ because this notebook runs from inside the notebooks/ folder
df = pd.read_csv('../data/scraped_healthcare_posts.csv')

# Check the shape — we expect 100 rows and 3 columns (title, post_url, content)
print(f"DataFrame shape: {df.shape}")

# Check for missing values — NLP functions will break on NaN inputs
print(f"\nMissing values per column:")
print(df.isnull().sum())

# Preview the first 3 rows to make sure the data looks right
print("\nFirst 3 posts:")
print(df.head(3))

print("\nData loaded successfully")

## Algorithm 1: LSA Summarizer

**LSA (Latent Semantic Analysis)** is an extractive summarization method — it picks the most important sentences directly from the original text rather than generating new ones.

**How it works in plain English:**
1. It breaks the text into individual sentences
2. It builds a matrix of which words appear in which sentences
3. It uses a linear algebra technique called SVD (Singular Value Decomposition) to figure out which sentences capture the most meaning
4. It returns the top-ranked sentences as the summary

**Why LSA for this project:**
- Free and runs locally — no API key or internet needed
- Great for factual content like forum posts where the original wording matters
- Fast enough to process 100 posts in seconds

**Limitation:** Because it extracts whole sentences, the summaries can sound choppy or disconnected — especially when sentences are pulled from different parts of a long post. We'll compare this with GPT's smoother abstractive summaries later.

In [ ]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer
import nltk

# Download the tokenizer data that sumy needs to split text into sentences
# Python 3.13 specifically requires punkt_tab or the tokenizer will throw
# a LookupError — we download both to be safe across Python versions
nltk.download('punkt')
nltk.download('punkt_tab')


def summarize_lsa(text, num_sentences=2):
    """
    Summarize a piece of text using LSA (extractive summarization).
    Returns a plain string summary with no line breaks.
    """
    try:
        # If the text is too short, there's nothing to summarize
        # Just return the first 300 characters as-is
        if len(text.split()) < 50:
            return text[:300]

        # Parse the text into sentences so LSA can rank them
        parser = PlaintextParser.from_string(text, Tokenizer("english"))

        # Create the LSA summarizer and run it
        summarizer = LsaSummarizer()
        summary_sentences = summarizer(parser.document, num_sentences)

        # Join the top sentences into a single string with no line breaks
        summary = " ".join(str(sentence) for sentence in summary_sentences)

        # If the summarizer returned nothing, fall back to first 300 chars
        if not summary.strip():
            return text[:300]

        return summary

    except Exception as e:
        # If anything goes wrong, return first 300 chars as a safe fallback
        print(f"  LSA error: {e}")
        return text[:300]


# --- Test LSA on the first 5 posts to make sure it works ---
print("Testing LSA on first 5 posts:\n")

for i in range(5):
    title = df.iloc[i]['title']
    content = df.iloc[i]['content']

    # Handle missing content — use empty string if NaN
    if pd.isna(content):
        content = ""

    summary = summarize_lsa(content)
    print(f"Post {i+1}: {title}")
    print(f"Summary: {summary[:200]}...\n")

print("LSA test complete")

In [ ]:
from tqdm import tqdm

# --- Run LSA summarization on all 100 posts ---
print("Running LSA on all 100 posts...\n")

lsa_summaries = []

for i in tqdm(range(len(df)), desc="LSA Summarization"):
    content = df.iloc[i]['content']

    # Handle missing content — use empty string if NaN
    if pd.isna(content):
        content = ""

    # Generate the LSA summary for this post
    summary = summarize_lsa(content)

    # Make sure summary is a clean string with no line breaks
    summary = summary.replace("\n", " ").strip()

    # Final safety check — if summary is empty, use first 300 chars
    if not summary:
        summary = content[:300]

    lsa_summaries.append(summary)

# Store the summaries in a new column
df['lsa_summary'] = lsa_summaries

print("\nLSA complete — sample output:")
print(df[['title', 'lsa_summary']].head(3))

## LSA Results Preview

The LSA summaries above are **extractive** — they are actual sentences pulled directly from the original posts and comments. You may notice they can sound choppy or lack smooth transitions, since the sentences were selected independently based on their importance score rather than written as a cohesive paragraph.

In the next section, we'll apply **OpenAI's gpt-4o-mini** to generate abstractive summaries of the same posts. This will let us compare:
- **LSA (extractive):** faithful to original wording, but can be disjointed
- **GPT (abstractive):** smoother and more readable, but rephrases the original text

Both approaches have their strengths — comparing them side by side will show which works better for healthcare forum data.